# Разработка модели для прогнозирования оттока клиентов

In [1]:
# ruff: noqa: E402

import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve() / "src"))

import warnings
warnings.filterwarnings("ignore")

import mlflow
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

from ml_churn.data import load_data, clean, encode, make_split
from ml_churn.model import build_lgbm
from ml_churn.evaluate import compute_metrics, summary_table
from ml_churn.tune import run_optuna_study

DATA_PATH = Path("../data/raw/telco-customer-churn-kaggle-dataset_2026-05-27.csv")
RANDOM_STATE = 42

In [2]:
mlflow.set_tracking_uri("http://localhost:5001")
mlflow.set_experiment("churn-prediction-notebook")

<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1780155567183, experiment_id='2', last_update_time=1780155567183, lifecycle_stage='active', name='churn-prediction-notebook', tags={}, trace_location=None, workspace='default'>

## 1. Загрузка данных

In [3]:
df_raw = load_data(DATA_PATH)
print(f"Shape: {df_raw.shape}")
df_raw.head()

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df_raw.info()
df_raw.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


## 2. Очистка данных

In [5]:
df_clean = clean(df_raw)
print(f"После очистки: {df_clean.shape}")
print(f"\nРаспределение Churn:\n{df_clean['Churn'].value_counts(normalize=True).round(3)}")

После очистки: (7032, 20)

Распределение Churn:
Churn
0    0.734
1    0.266
Name: proportion, dtype: float64


In [6]:
df = encode(df_clean)
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,1,0,1,0,1,0,0,2,0,0,0,0,0,1,2,29.85,29.85,0
1,1,0,0,0,34,1,0,0,2,0,2,0,0,0,1,0,3,56.95,1889.50,0
2,1,0,0,0,2,1,0,0,2,2,0,0,0,0,0,1,3,53.85,108.15,1
3,1,0,0,0,45,0,1,0,2,0,2,2,0,0,1,0,0,42.30,1840.75,0
4,0,0,0,0,2,1,0,1,0,0,0,0,0,0,0,1,2,70.70,151.65,1


## 3. Разделение на train / test

In [7]:
X_train, X_test, y_train, y_test = make_split(df, random_state=RANDOM_STATE)

print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
print(f"Churn rate train: {y_train.mean():.3f} | test: {y_test.mean():.3f}")

Train: 5625 | Test: 1407
Churn rate train: 0.266 | test: 0.266


## 4. Эксперименты с моделями

In [8]:
experiments = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "Decision Tree": Pipeline([
        ("clf", DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE)),
    ]),
    "Random Forest": Pipeline([
        ("clf", RandomForestClassifier(n_estimators=200, max_depth=8, random_state=RANDOM_STATE)),
    ]),
    "LightGBM": Pipeline([
        ("clf", build_lgbm(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=RANDOM_STATE)),
    ]),
}

results = {}
for name, model in experiments.items():
    with mlflow.start_run(run_name=name):
        model.fit(X_train, y_train)
        metrics = compute_metrics(model, X_test, y_test, X_train, y_train)
        results[name] = metrics
        mlflow.log_metrics(metrics=metrics)

print("Done")

🏃 View run Logistic Regression at: http://localhost:5001/#/experiments/2/runs/f8f4b0fafeb34842bcc8c6274322ceac
🧪 View experiment at: http://localhost:5001/#/experiments/2
🏃 View run Decision Tree at: http://localhost:5001/#/experiments/2/runs/1cb1d2d4115b429cbd0ec5c33b54a285
🧪 View experiment at: http://localhost:5001/#/experiments/2
🏃 View run Random Forest at: http://localhost:5001/#/experiments/2/runs/407a45d413fd49afbbc40bb23507ed7f
🧪 View experiment at: http://localhost:5001/#/experiments/2
🏃 View run LightGBM at: http://localhost:5001/#/experiments/2/runs/c0687e4e24b14b289f77de5c30697586
🧪 View experiment at: http://localhost:5001/#/experiments/2
Done


In [9]:
summary_table(results)

,accuracy,roc_auc,f1,precision,recall,cv_roc_auc
Model,,,,,,
Logistic Regression,0.7939,0.8345,0.5927,0.6243,0.5642,0.8452
Decision Tree,0.7783,0.8086,0.5517,0.5963,0.5134,0.8141
Random Forest,0.7918,0.8337,0.5540,0.6431,0.4866,0.8464
LightGBM,0.7960,0.8366,0.5798,0.6408,0.5294,0.8447


In [10]:
best_name = max(results, key=lambda m: results[m]["roc_auc"])
print(f"Лучшая модель по ROC-AUC: {best_name}")
print()
print(classification_report(y_test, experiments[best_name].predict(X_test), target_names=["No Churn", "Churn"]))

Лучшая модель по ROC-AUC: LightGBM

              precision    recall  f1-score   support

    No Churn       0.84      0.89      0.87      1033
       Churn       0.64      0.53      0.58       374

    accuracy                           0.80      1407
   macro avg       0.74      0.71      0.72      1407
weighted avg       0.79      0.80      0.79      1407



## 6. Подбор гиперпараметров optuna

In [ ]:
with mlflow.start_run(run_name="lgbm_tuned"):
    study = run_optuna_study(X_train, y_train, n_trials=10, random_state=RANDOM_STATE)

    print(f"\nЛучший cv_roc_auc: {study.best_value:.4f}")
    print(f"Лучшие параметры:\n{study.best_params}")

    lgbm_tuned = build_lgbm(**study.best_params, random_state=RANDOM_STATE)
    lgbm_tuned.fit(X_train, y_train)

    metrics = compute_metrics(lgbm_tuned, X_test, y_test, X_train, y_train)
    mlflow.log_metrics(metrics=metrics)

Best trial: 9. Best value: 0.848559: 100%|██████████| 10/10 [01:27<00:00,  8.74s/it]



Лучший cv_roc_auc: 0.8486
Лучшие параметры:
{'n_estimators': 412, 'learning_rate': 0.030816017044468066, 'max_depth': 3, 'num_leaves': 60, 'min_child_samples': 36, 'subsample': 0.864803089169032, 'colsample_bytree': 0.8187787356776066, 'reg_alpha': 0.9658611176861261, 'reg_lambda': 0.0001778010520878397}
🏃 View run lgbm_tuned at: http://localhost:5001/#/experiments/2/runs/59302b900cfb46b1a169b252824c78e3
🧪 View experiment at: http://localhost:5001/#/experiments/2


In [13]:
summary_table({
    "LightGBM (baseline)": results["LightGBM"],
    "LightGBM (Optuna)": metrics,
})

,accuracy,roc_auc,f1,precision,recall,cv_roc_auc
Model,,,,,,
LightGBM (baseline),0.7960,0.8366,0.5798,0.6408,0.5294,0.8447
LightGBM (Optuna),0.7982,0.8378,0.5836,0.6461,0.5321,0.8473


In [14]:
print(classification_report(y_test, lgbm_tuned.predict(X_test), target_names=["No Churn", "Churn"]))

              precision    recall  f1-score   support

    No Churn       0.84      0.89      0.87      1033
       Churn       0.65      0.53      0.58       374

    accuracy                           0.80      1407
   macro avg       0.74      0.71      0.73      1407
weighted avg       0.79      0.80      0.79      1407

